In [10]:
import sagemaker
from sagemaker.sklearn.estimator import SKLearn

# Create SageMaker session
sagemaker_session = sagemaker.Session()

# Get IAM execution role automatically (works inside Studio)
role = sagemaker.get_execution_role()

# S3 paths
train_input = "s3://my-avazu-bucket-12345/session_features/"
output_path = "s3://my-avazu-bucket-12345/model_output/"

# Define Scikit-Learn training job
estimator = SKLearn(
    entry_point="train_isolation_forest.py",
    framework_version="1.2-1",
    instance_type="ml.m5.large",     # ✅ valid training instance
    instance_count=1,
    role=role,
    base_job_name="click-fraud-train",
    output_path=output_path,
    hyperparameters={"contamination": 0.02},
)
estimator.fit({"train": train_input})



INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: click-fraud-train-2025-10-26-16-27-23-879


2025-10-26 16:27:25 Starting - Starting the training job...
2025-10-26 16:27:39 Starting - Preparing the instances for training...
2025-10-26 16:28:26 Downloading - Downloading the training image......
2025-10-26 16:29:27 Training - Training image download completed. Training in progress.../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-10-26 16:29:31,628 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2025-10-26 16:29:31,632 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2025-10-26 16:29:31,635 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-10-26 16:29:31,651 sagemaker_skle

In [12]:
from sagemaker.sklearn.model import SKLearnModel
import sagemaker

# Get your execution role
role = sagemaker.get_execution_role()

# Create model object
model = SKLearnModel(
    model_data="s3://my-avazu-bucket-12345/model_output/click-fraud-train-2025-10-26-16-27-23-879/output/model.tar.gz",
    role="arn:aws:iam::478614262887:role/service-role/AmazonSageMaker-ExecutionRole-20251013T212419",
    entry_point="inference.py",
    framework_version="1.2-1"
)

# Deploy endpoint
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",  # You can use ml.t3.medium if cost is a concern
    endpoint_name="click-fraud-detector-3"
)


INFO:sagemaker:Creating model with name: sagemaker-scikit-learn-2025-10-26-16-31-54-229
INFO:sagemaker:Creating endpoint-config with name click-fraud-detector-3
INFO:sagemaker:Creating endpoint with name click-fraud-detector-3


------!

In [13]:
import boto3
import json

# create a runtime client
runtime = boto3.client("sagemaker-runtime")

# define the endpoint name
endpoint_name = "click-fraud-detector-3"

# Example: Human-like click pattern (normal)
human = [5, 1, 0.2, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps({"features": human})
)

result = json.loads(response["Body"].read().decode())
print(result)


{'is_fraud': 0, 'anomaly_score': 0.18774173409725592}


In [14]:
bot = [50, 45, 0.9, 20, 10, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5]

response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Body=json.dumps({"features": bot})
)

print(json.loads(response["Body"].read().decode()))


{'is_fraud': 1, 'anomaly_score': -0.19003739258511443}


In [20]:
pip install --upgrade scikit-learn==1.7.0 joblib pandas numpy boto3 streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 129.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 188.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 20.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 57.3 MB/s  0:00:00ta 0:00:02
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: joblib━━━━━━━━━━━━━━━━━━ 0/7 [numpy]
    Found existing installation: joblib 1.5.1 0/7 [numpy]
    Uninstalling joblib-1.5.1:━━━━━━━━━━━━━━ 0/7 [numpy]
      Successfully uninstalled joblib-1.5.1━ 0/7 [numpy]
  Attempting uninstall: pandas0m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/7 [joblib]
    Found existing installation: pandas 2.3.1━━━━━━━━━━━━━━━━━ 1/7 [joblib]
    Uninstalling pandas-2.3.1:━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/7 [pandas]
      Successfully uninstalled pandas-2.3.1━━━━━━━━━━━━━━━━━